# MVP - Classificação de Faixa de Preço de Combustível\n
\n
## Contexto\n
Este notebook utiliza o dataset **Global Fuel Prices: 100 Years (1924-2024)** para resolver um problema de classificação supervisionada: prever `price_tier`.

## 1. Carga de dados por URL (requisito do MVP)\n
Substitua a URL abaixo pelo RAW do seu repositório GitHub.

In [ ]:
import pandas as pd
DATA_URL = 'https://raw.githubusercontent.com/AngeloSouzaOliveira/API-Fuel-Insight-MVP/refs/heads/main/data/all_countries_combined.csv'
df = pd.read_csv(DATA_URL)
df.head()

## 2. Preparação de dados (holdout + transformação + pipelines)

In [ ]:
from sklearn.model_selection import train_test_split

features = [
    'year','region','subsidy_regime','is_oil_producer',
    'crude_oil_usd_per_barrel','tax_pct_of_pump_price','gasoline_real_2024usd'
]
target = 'price_tier'

df = df.dropna(subset=features + [target]).copy()
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## 3. Modelagem (KNN, Árvore, Naive Bayes e SVM) + otimização

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

num_features = ['year','is_oil_producer','crude_oil_usd_per_barrel','tax_pct_of_pump_price','gasoline_real_2024usd']
cat_features = ['region','subsidy_regime']

base_pre = ColumnTransformer([
    ('num', Pipeline([('scaler', StandardScaler())]), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

candidates = {
    'KNN': (KNeighborsClassifier(), {'clf__n_neighbors': [3,5,7,9]}),
    'DecisionTree': (DecisionTreeClassifier(random_state=42), {'clf__max_depth': [3,5,8,None], 'clf__min_samples_split': [2,5,10]}),
    'NaiveBayes': (GaussianNB(), {}),
    'SVM': (SVC(random_state=42), {
        'pre__num__scaler': [StandardScaler(), MinMaxScaler(), RobustScaler()],
        'clf__C': [0.5,1,5,10],
        'clf__kernel': ['linear','rbf']
    })
}

results = []
best_model = None
best_acc = -1

for name, (clf, grid) in candidates.items():
    pipe = Pipeline([('pre', base_pre), ('clf', clf)])
    gs = GridSearchCV(pipe, grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train, y_train)
    pred = gs.best_estimator_.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results.append((name, acc, gs.best_params_))
    if acc > best_acc:
        best_acc = acc
        best_model = gs.best_estimator_

results

## 4. Exportação do modelo vencedor

In [ ]:
import joblib

joblib.dump(best_model, 'fuel_price_tier_model.pkl')
best_acc

## 5. Análise final
Fuel-Insight-MVP é uma aplicação full stack com foco comercial que prevê faixas de preço de combustíveis usando machine learning. A plataforma oferece autenticação (login/cadastro), navegação SPA, predição unitária e em lote, analytics por região/país, alertas de risco, dashboard executivo com KPIs e módulo de mercado comparativo com gráfico (Brent, WTI, USD/BRL e outros ativos).

Seu objetivo é apoiar decisões estratégicas de preço e risco no setor de combustíveis, transformando dados históricos em insights práticos para operação, planejamento e governança de modelos (MLOps), com registro, ativação e monitoramento de versões do modelo.